# Week 8 Exercise: Multi-Agent Debate System for Price Band Classification

## Deliberative Multi-Agent Architecture: Debate + Judge Crew

This notebook implements a sophisticated multi-agent system where:
- **Advocate Agents** (Budget, Midrange, Premium) use specialized decision-making strategies to build evidence-based cases
- **Judge Agent** employs a rigorous evaluation rubric to assess argument quality
- **Interactive UI** shows real-time debate with color-coded agents

## 1. Imports and Setup

In [ ]:
# Core imports
import os
import json
import random
import logging
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, asdict
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# HuggingFace and OpenAI
from huggingface_hub import login
from openai import OpenAI

# Data analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Gradio for UI
import gradio as gr

# Week 6/7 imports - for Item class
import sys
sys.path.append('../week7')
from pricer.items import Item

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(message)s'
)

print("All imports successful")

In [ ]:
# Authenticate
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("HuggingFace authenticated")

# Initialize OpenAI client
openai_client = OpenAI()
print("OpenAI client initialized")

## 2. Configuration

In [ ]:
# Configuration
LITE_MODE = False  # Set to True for smaller dataset
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model configuration
GPT_MODEL = "gpt-4o-mini"  # Using frontier model
TEMPERATURE = 0.7  # Slightly creative for debate

# Judge decision thresholds
AMBIGUITY_MARGIN = 0.08  # If top two scores within this margin, consider ambiguous
MAX_CLARIFICATIONS = 1  # Only ask one clarification question

## 3. Load Data and Define Price Bands

In [ ]:
# Load preprocessed data from week 6
username = "ed-donner"  # Using the instructor's dataset
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items")
print(f"Loaded {len(val):,} validation items")
print(f"Loaded {len(test):,} test items")

In [ ]:
# Analyze price distribution and define bands
all_items = train + val + test
all_prices = [item.price for item in all_items]

# Use 33rd and 67th percentiles for balanced classes
percentile_33 = np.percentile(all_prices, 33.33)
percentile_67 = np.percentile(all_prices, 66.67)

BUDGET_MAX = percentile_33
MIDRANGE_MAX = percentile_67

print("Price Band Definitions:")
print(f"  Budget: $0 - ${BUDGET_MAX:.2f}")
print(f"  Midrange: ${BUDGET_MAX:.2f} - ${MIDRANGE_MAX:.2f}")
print(f"  Premium: ${MIDRANGE_MAX:.2f}+")


def get_price_band(price: float) -> str:
    """Classify price into budget, midrange, or premium"""
    if price <= BUDGET_MAX:
        return "budget"
    elif price <= MIDRANGE_MAX:
        return "midrange"
    else:
        return "premium"


# Show distribution
bands = [get_price_band(price) for price in all_prices]
band_counts = pd.Series(bands).value_counts()

print("\nClass Distribution:")
for band, count in band_counts.items():
    percentage = (count / len(bands)) * 100
    print(f"  {band}: {count:,} ({percentage:.1f}%)")

## 4. Data Structures

In [ ]:
@dataclass
class ProductInput:
    """Strict input contract for debate"""
    title: str
    category: str
    brand: Optional[str]
    condition: Optional[str]
    description: Optional[str]
    price: Optional[float] = None  # May be missing
    warranty: Optional[str] = None
    age: Optional[str] = None
    
    @classmethod
    def from_item(cls, item: Item) -> 'ProductInput':
        """Create ProductInput from Item object with schema-safe fallbacks"""
        return cls(
            title=getattr(item, "title", "Unknown item"),
            category=getattr(item, "category", "Unknown") or "Unknown",
            brand=getattr(item, "brand", None),
            condition=getattr(item, "condition", None),
            description=(
                getattr(item, "description", None)
                or getattr(item, "summary", None)
                or getattr(item, "full", None)
            ),
            price=None,  # Hide price from agents initially
            warranty=getattr(item, "warranty", None),
            age=getattr(item, "age", None)
        )
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for LLM"""
        return {k: v for k, v in asdict(self).items() if v is not None}


@dataclass
class Argument:
    """An advocate's argument"""
    agent_name: str
    price_band: str
    reasoning: str
    evidence: List[str]
    confidence: float
    acknowledges_weaknesses: bool
    


@dataclass
class JudgeScore:
    """Judge's evaluation of an argument"""
    agent_name: str
    evidence_grounding: float  # 0-1
    internal_consistency: float  # 0-1
    domain_plausibility: float  # 0-1
    counterpoint_handling: float  # 0-1
    confidence_calibration: float  # 0-1
    total_score: float
    


@dataclass
class JudgeDecision:
    """Final judge decision"""
    final_label: str
    explanation: str
    scores: List[JudgeScore]
    is_ambiguous: bool
    clarification_question: Optional[str] = None


print("Data structures defined")

## 5. Agent Base Class

In [ ]:
class Agent:
    """Base agent class with colored logging"""
    
    # ANSI color codes
    RED = '\033[31m'
    GREEN = '\033[32m'
    YELLOW = '\033[33m'
    BLUE = '\033[34m'
    MAGENTA = '\033[35m'
    CYAN = '\033[36m'
    WHITE = '\033[37m'
    BG_BLACK = '\033[40m'
    RESET = '\033[0m'
    BOLD = '\033[1m'
    
    def __init__(self, name: str, color: str):
        self.name = name
        self.color = color
        
    def log(self, message: str):
        """Log message with agent color"""
        color_code = self.BG_BLACK + self.color + self.BOLD
        formatted = f"{color_code}[{self.name}]{self.RESET} {message}"
        logging.info(formatted)
        return formatted  # Return for UI display


print("Agent base class defined")

## 6. Advocate Agents with Specialized Decision-Making

In [ ]:
class AdvocateAgent(Agent):
    """Base class for advocate agents with specialized prompts"""
    
    def __init__(self, name: str, price_band: str, color: str, client: OpenAI):
        super().__init__(name, color)
        self.price_band = price_band
        self.client = client
    
    def get_system_prompt(self) -> str:
        """Override in subclasses for specialized prompts"""
        raise NotImplementedError("Subclasses must implement get_system_prompt")
        
    def build_argument(self, product: ProductInput) -> Argument:
        """Build evidence-based argument using specialized strategy"""
        self.log(f"Analyzing product for {self.price_band} classification...")
        
        # Get specialized system prompt
        system_prompt = self.get_system_prompt()
        
        # Create user prompt
        user_prompt = f"""Product to classify:
{json.dumps(product.to_dict(), indent=2)}

Using your specialized decision-making strategy, build the strongest possible evidence-based case for the {self.price_band.upper()} price band."""
        
        # Call LLM
        response = self.client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=TEMPERATURE,
            response_format={"type": "json_object"}
        )
        
        # Parse response
        result = json.loads(response.choices[0].message.content)
        print(result)  # Debugging output
        
        # Create argument
        argument = Argument(
            agent_name=self.name,
            price_band=self.price_band,
            reasoning=result["reasoning"],
            evidence=result["evidence"],
            confidence=result["confidence"] / 100.0,
            acknowledges_weaknesses=result.get("acknowledges_weaknesses", False)
        )
        
        # Log argument summary
        self.log(f"Case strength: {argument.confidence:.0%}")
        
        return argument


print("Base advocate agent defined")

In [ ]:
class BudgetAdvocateAgent(AdvocateAgent):
    """Specialist in identifying budget-friendly products"""
    
    def __init__(self, client: OpenAI):
        super().__init__("BudgetAdvocate", "budget", Agent.GREEN, client)
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, an expert in identifying budget-friendly products and value deals.

MISSION: Build the strongest case that this product belongs in BUDGET ($0-${BUDGET_MAX:.2f}).

DECISION-MAKING STRATEGY:

1. CATEGORY ANALYSIS (Weight: High)
   • Generic/commodity categories → Strong budget signal
   • Mass-market everyday items → Budget indicator
   • Specialty/niche categories → Weakness

2. BRAND POSITIONING (Weight: Critical)
   • No brand/generic → Strong budget
   • Value brands (e.g., Amazon Basics, generic) → Strong budget
   • Mid-tier brands → Moderate weakness
   • Premium brands (Apple, Sony, Samsung flagship) → Major weakness

3. CONDITION IMPACT (Weight: High)
   • "For parts or not working" → Strong budget
   • "Used" with wear → Strong budget
   • "Refurbished" → Moderate budget
   • "New" → Check brand/features carefully

4. FEATURE SET (Weight: Medium)
   • Older generation (e.g., "2015", "Series 5") → Budget
   • Basic/entry-level models → Budget
   • Limited specs mentioned → Budget
   • Premium features ("Pro", "Max", "Ultra") → Weakness

5. MARKET SIGNALS (Weight: Medium)
   • Description terms: "affordable", "basic", "starter" → Budget
   • Small storage (16GB, 32GB) → Budget
   • No special features mentioned → Budget

BEHAVIORAL RULES:
Cite ONLY facts from product data - no invented information
Acknowledge ALL weaknesses honestly (premium brand, new condition, high-end features)
Calibrate confidence:
  - 80-95%: Multiple strong budget signals, no major contradictions
  - 60-79%: Clear budget indicators but some weaknesses
  - 40-59%: Mixed signals, requires careful argument
  - 20-39%: Weak case, major contradictions
If evidence is weak, be intellectually honest

OUTPUT (JSON):
{{
    "reasoning": "Structured argument with category, brand, condition, features analysis",
    "evidence": ["Specific fact from product", "Another concrete fact", "Third verifiable point"],
    "confidence": 75,
    "acknowledges_weaknesses": true,
    "weaknesses": "Specific contradictory evidence like premium brand or new condition"
}}"""


print("Budget advocate defined")

In [ ]:
class MidrangeAdvocateAgent(AdvocateAgent):
    """Expert in mainstream market positioning"""
    
    def __init__(self, client: OpenAI):
        super().__init__("MidrangeAdvocate", "midrange", Agent.YELLOW, client)
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, an expert in mainstream market products with balanced value and quality.

MISSION: Build the strongest case that this product belongs in MIDRANGE (${BUDGET_MAX:.2f}-${MIDRANGE_MAX:.2f}).

DECISION-MAKING STRATEGY:

1. MAINSTREAM POSITIONING (Weight: Critical)
   • Popular product categories → Midrange indicator
   • Mass-market appeal → Midrange signal
   • Neither too basic NOR too exotic → Sweet spot

2. BRAND SWEET SPOT (Weight: High)
   • Established consumer brands (LG, HP, Dell) → Strong midrange
   • Mid-tier lines from premium brands → Midrange
   • Generic brands → Weakness (too budget)
   • Luxury brands with "new" condition → Weakness (too premium)

3. BALANCED CONDITION (Weight: High)
   • "New" mainstream products → Strong midrange
   • "Refurbished" premium brands → Could be midrange
   • "Used" mid-tier in good condition → Midrange
   • "For parts" → Weakness (budget)
   • "New" luxury flagship → Weakness (premium)

4. FEATURE BALANCE (Weight: Medium)
   • Standard current-gen features → Midrange
   • Last year's mainstream models → Midrange
   • Moderate specs (128GB, standard RAM) → Midrange
   • Too basic OR cutting-edge → Weakness

5. GOLDILOCKS PRINCIPLE (Weight: Medium)
   • Not entry-level, not flagship → Midrange
   • "Standard" or base models of good brands → Midrange
   • Balanced features-to-price expectations → Midrange

BEHAVIORAL RULES:
Cite ONLY product facts - no market assumptions
Acknowledge weaknesses pointing to EITHER budget OR premium
Calibrate confidence:
  - 80-95%: Clear mainstream positioning, balanced indicators
  - 60-79%: Good midrange signals with minor extremes
  - 40-59%: Mixed signals, could go either way
  - 20-39%: Product clearly skews to budget or premium
Midrange is about BALANCE - argue why product is "just right"

OUTPUT (JSON):
{{
    "reasoning": "Show how product balances between budget and premium",
    "evidence": ["Mainstream indicator", "Balanced feature", "Mid-tier brand signal"],
    "confidence": 70,
    "acknowledges_weaknesses": true,
    "weaknesses": "Factors suggesting budget OR premium classification"
}}"""


print("Midrange advocate defined")

In [ ]:
class PremiumAdvocateAgent(AdvocateAgent):
    """Specialist in luxury and high-end products"""
    
    def __init__(self, client: OpenAI):
        super().__init__("PremiumAdvocate", "premium", Agent.MAGENTA, client)
    
    def get_system_prompt(self) -> str:
        return f"""You are {self.name}, an expert in identifying premium, luxury, and high-value products.

MISSION: Build the strongest case that this product belongs in PREMIUM (${MIDRANGE_MAX:.2f}+).

DECISION-MAKING STRATEGY:

1. LUXURY BRAND RECOGNITION (Weight: Critical)
   • Premium brands (Apple, Rolex, Gucci, Bang & Olufsen) → Strong premium
   • Professional/prosumer brands → Premium indicator
   • Generic/value brands → Major weakness

2. FLAGSHIP POSITIONING (Weight: Critical)
   • "Pro", "Max", "Ultra", "Premium" in title → Strong premium
   • Latest generation flagship models → Premium
   • Professional/business lines → Premium
   • Entry-level or mid-tier lines → Weakness

3. LUXURY VALUE RETENTION (Weight: High)
   • "New" premium brand → Strong premium
   • "Used" luxury items (retain value) → Still premium
   • "Refurbished" flagship → Can be premium
   • "For parts" or severely damaged → Weakness

4. PREMIUM FEATURES (Weight: High)
   • High storage (512GB, 1TB+) → Premium
   • Professional features/capabilities → Premium
   • Advanced specs mentioned → Premium
   • Basic/standard features only → Weakness

5. CATEGORY PRESTIGE (Weight: Medium)
   • Luxury categories (jewelry, designer items) → Premium
   • Professional equipment → Premium
   • Commodity categories → Needs strong brand/features

BEHAVIORAL RULES:
Cite ONLY product facts - premium requires EVIDENCE
Acknowledge ALL weaknesses (condition issues, budget brands, basic features)
Calibrate confidence:
  - 85-95%: Luxury brand + flagship + new/excellent condition
  - 70-84%: Strong premium signals (brand OR flagship + good condition)
  - 50-69%: Some premium indicators but significant weaknesses
  - 20-49%: Weak premium case, mostly budget/midrange signals
Premium classification requires concrete luxury indicators

OUTPUT (JSON):
{{
    "reasoning": "Demonstrate premium through brand, flagship status, or professional features",
    "evidence": ["Luxury brand fact", "Flagship indicator", "Premium feature"],
    "confidence": 85,
    "acknowledges_weaknesses": true,
    "weaknesses": "Factors like damage, age, or mid-tier positioning"
}}"""


print("Premium advocate defined")

## 7. Judge Agent with Rigorous Evaluation

In [ ]:
class JudgeAgent(Agent):
    """Impartial judge with systematic evaluation rubric"""
    
    def __init__(self, client: OpenAI):
        super().__init__("Judge", Agent.CYAN)
        self.client = client
    
    def get_judge_prompt(self) -> str:
        """Get specialized judge evaluation prompt"""
        return f"""You are an impartial Judge evaluating price band classification arguments.

Price bands:
• Budget: $0 - ${BUDGET_MAX:.2f}
• Midrange: ${BUDGET_MAX:.2f} - ${MIDRANGE_MAX:.2f}
• Premium: ${MIDRANGE_MAX:.2f}+

EVALUATION METHODOLOGY:

You MUST score each argument on 5 dimensions (0.0-1.0 scale):

1. EVIDENCE GROUNDING (0-1.0)
   • 0.9-1.0: Every claim directly maps to quoted product facts
   • 0.7-0.8: Most claims grounded, minor interpretation
   • 0.5-0.6: Some grounded evidence, some unsupported claims
   • 0.0-0.4: Invented facts, poor grounding

2. INTERNAL CONSISTENCY (0-1.0)
   • 0.9-1.0: No contradictions, logic flows perfectly
   • 0.7-0.8: Mostly consistent, minor logical gaps
   • 0.5-0.6: Some contradictions or unclear reasoning
   • 0.0-0.4: Major contradictions, incoherent

3. DOMAIN PLAUSIBILITY (0-1.0)
   • 0.9-1.0: Category + brand + condition logic perfect
   • 0.7-0.8: Generally plausible market reasoning
   • 0.5-0.6: Questionable market assumptions
   • 0.0-0.4: Implausible domain logic

4. COUNTERPOINT HANDLING (0-1.0)
   • 0.9-1.0: Acknowledges ALL weaknesses honestly
   • 0.7-0.8: Acknowledges main weaknesses
   • 0.5-0.6: Partial acknowledgment
   • 0.0-0.4: Ignores obvious contradictions

5. CONFIDENCE CALIBRATION (0-1.0)
   • 0.9-1.0: Confidence perfectly matches evidence strength
   • 0.7-0.8: Confidence reasonable given evidence
   • 0.5-0.6: Over/under confident relative to evidence
   • 0.0-0.4: Severely miscalibrated confidence

DECISION RULES:
Total score = sum of 5 dimensions (max 5.0)
Winner = HIGHEST total score (NOT majority vote)
If top two scores within {AMBIGUITY_MARGIN}, mark ambiguous
If ambiguous AND critical info missing, ask ONE question
Evaluate ARGUMENT QUALITY, not just agreement

OUTPUT (JSON):
{{
    "scores": [
        {{
            "agent_name": "BudgetAdvocate",
            "evidence_grounding": 0.85,
            "internal_consistency": 0.90,
            "domain_plausibility": 0.75,
            "counterpoint_handling": 0.80,
            "confidence_calibration": 0.85,
            "total_score": 4.15
        }},
        ... (for each agent)
    ],
    "final_label": "budget",
    "explanation": "Detailed reasoning showing why winner had best argument quality",
    "is_ambiguous": false,
    "clarification_question": null
}}"""
        
    def evaluate_arguments(
        self, 
        product: ProductInput, 
        arguments: List[Argument],
        clarification_answer: Optional[str] = None
    ) -> JudgeDecision:
        """Evaluate all arguments and make final decision"""
        self.log("Evaluating arguments with systematic rubric...")
        
        # Build clarification context
        clarification_context = ""
        if clarification_answer:
            clarification_context = f"\n\nADDITIONAL INFORMATION PROVIDED:\n{clarification_answer}"
        
        # Create user prompt
        arguments_text = "\n\n".join([
            f"""=== ARGUMENT FROM {arg.agent_name} (for {arg.price_band.upper()}) ===
Reasoning: {arg.reasoning}
Evidence: {json.dumps(arg.evidence, indent=2)}
Confidence: {arg.confidence:.0%}
Acknowledges weaknesses: {arg.acknowledges_weaknesses}"""
            for arg in arguments
        ])
        
        user_prompt = f"""PRODUCT DATA:
{json.dumps(product.to_dict(), indent=2)}

ARGUMENTS TO EVALUATE:
{arguments_text}{clarification_context}

Using your evaluation rubric, score each argument on all 5 dimensions and determine the winner."""
        
        # Call LLM
        response = self.client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": self.get_judge_prompt()},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.3,  # More deterministic for judging
            response_format={"type": "json_object"}
        )
        
        # Parse response
        result = json.loads(response.choices[0].message.content)
        print(result)  # Debugging output
        
        # Create scores
        scores = [
            JudgeScore(
                agent_name=s["agent_name"],
                evidence_grounding=s["evidence_grounding"],
                internal_consistency=s["internal_consistency"],
                domain_plausibility=s["domain_plausibility"],
                counterpoint_handling=s["counterpoint_handling"],
                confidence_calibration=s["confidence_calibration"],
                total_score=s["total_score"]
            )
            for s in result["scores"]
        ]
        
        # Create decision
        decision = JudgeDecision(
            final_label=result["final_label"],
            explanation=result["explanation"],
            scores=scores,
            is_ambiguous=result.get("is_ambiguous", False),
            clarification_question=result.get("clarification_question")
        )
        
        # Log decision
        self.log(f"VERDICT: {decision.final_label.upper()}")
        
        return decision


print("Judge agent defined")

## 8. Debate Orchestrator

In [ ]:
class DebateOrchestrator:
    """Orchestrates the multi-agent debate process"""
    
    def __init__(self, client: OpenAI):
        self.client = client
        self.budget_advocate = BudgetAdvocateAgent(client)
        self.midrange_advocate = MidrangeAdvocateAgent(client)
        self.premium_advocate = PremiumAdvocateAgent(client)
        self.judge = JudgeAgent(client)
        self.debate_log = []
        
    def run_debate(
        self, 
        product: ProductInput,
        clarification_answer: Optional[str] = None
    ) -> Tuple[JudgeDecision, List[Argument]]:
        """Run complete debate with all agents"""
        self.debate_log = []
        
        # Log start
        self.add_to_log("SYSTEM", f"Starting debate for: {product.title}", "white")
        
        # Each advocate builds their argument
        arguments = []
        
        for advocate in [self.budget_advocate, self.midrange_advocate, self.premium_advocate]:
            arg = advocate.build_argument(product)
            arguments.append(arg)
            
            # Add to debate log
            self.add_to_log(
                advocate.name,
                f"**Argues for {arg.price_band.upper()}** (Confidence: {arg.confidence:.0%})\n\n{arg.reasoning}",
                advocate.color
            )
        
        # Judge evaluates
        decision = self.judge.evaluate_arguments(product, arguments, clarification_answer)
        
        # Add judge decision to log
        scores_text = "\n".join([
            f"  • **{s.agent_name}**: {s.total_score:.2f}/5.0 "
            f"(EG:{s.evidence_grounding:.2f} IC:{s.internal_consistency:.2f} "
            f"DP:{s.domain_plausibility:.2f} CH:{s.counterpoint_handling:.2f} "
            f"CC:{s.confidence_calibration:.2f})"
            for s in sorted(decision.scores, key=lambda x: x.total_score, reverse=True)
        ])
        
        judge_message = f"""**FINAL VERDICT: {decision.final_label.upper()}**

**Detailed Scores:**
{scores_text}

**Reasoning:**
{decision.explanation}"""
        
        if decision.clarification_question:
            judge_message += f"\n\n**Clarification Needed:**\n{decision.clarification_question}"
        
        self.add_to_log("Judge", judge_message, self.judge.color)
        
        return decision, arguments
    
    def add_to_log(self, agent: str, message: str, color: str):
        """Add message to debate log"""
        self.debate_log.append({
            "agent": agent,
            "message": message,
            "color": color
        })


print("Debate orchestrator defined")

## 9. Multi-Round Debate Orchestrator with Critiques

In [ ]:
class MultiRoundDebateOrchestrator:
    """Enhanced orchestrator with multi-round debate and critiques"""
    
    def __init__(self, client: OpenAI, max_rounds: int = 3):
        self.client = client
        self.budget_advocate = BudgetAdvocateAgent(client)
        self.midrange_advocate = MidrangeAdvocateAgent(client)
        self.premium_advocate = PremiumAdvocateAgent(client)
        self.judge = JudgeAgent(client)
        self.debate_log = []
        self.max_rounds = max_rounds
        
    def run_multi_round_debate(
        self, 
        product: ProductInput,
        clarification_answer: Optional[str] = None
    ) -> Tuple[JudgeDecision, List[Dict]]:
        """Run multi-round debate with back-and-forth arguments"""
        self.debate_log = []
        conversation_history = []
        
        # Log start
        self.add_to_log("SYSTEM", f"Starting multi-round debate for: {product.title}", "white")
        
        advocates = [self.budget_advocate, self.midrange_advocate, self.premium_advocate]
        all_arguments = []
        
        # ROUND 1: Initial arguments
        print(f"\n{'='*80}")
        print("ROUND 1: Initial Arguments")
        print(f"{'='*80}\n")
        
        round_1_args = []
        for advocate in advocates:
            arg = self._make_initial_argument(advocate, product)
            round_1_args.append(arg)
            all_arguments.append({
                "round": 1,
                "agent": advocate.name,
                "type": "initial_argument",
                "argument": arg
            })
            
            # Add to conversation history
            conversation_history.append({
                "speaker": advocate.name,
                "message": f"I argue for {arg.price_band.upper()}: {arg.reasoning}",
                "evidence": arg.evidence
            })
            
            # Log
            arg_msg = f"**Round 1 - Initial Argument for {arg.price_band.upper()}**\n\n"
            arg_msg += f"Reasoning: {arg.reasoning}\n\n"
            arg_msg += f"Evidence:\n" + "\n".join([f"- {e}" for e in arg.evidence])
            arg_msg += f"\n\nConfidence: {arg.confidence:.0%}"
            self.add_to_log(advocate.name, arg_msg, advocate.color)
        
        # ROUNDS 2-N: Critiques and rebuttals
        for round_num in range(2, self.max_rounds + 1):
            print(f"\n{'='*80}")
            print(f"ROUND {round_num}: Critiques and Rebuttals")
            print(f"{'='*80}\n")
            
            round_args = []
            for i, advocate in enumerate(advocates):
                # Get other advocates' arguments
                other_args = [arg for j, arg in enumerate(round_1_args) if j != i]
                
                # Generate critique and rebuttal
                critique = self._make_critique_and_rebuttal(
                    advocate, 
                    product, 
                    round_1_args[i],  # Own argument
                    other_args,  # Others' arguments
                    conversation_history
                )
                
                round_args.append(critique)
                all_arguments.append({
                    "round": round_num,
                    "agent": advocate.name,
                    "type": "critique_rebuttal",
                    "content": critique
                })
                
                # Add to conversation history
                conversation_history.append({
                    "speaker": advocate.name,
                    "message": critique
                })
                
                # Log
                critique_msg = f"**Round {round_num} - Critique & Rebuttal**\n\n{critique}"
                self.add_to_log(advocate.name, critique_msg, advocate.color)
            
            # Check if judge wants to intervene
            should_stop = self._check_judge_intervention(
                product, 
                conversation_history,
                round_num
            )
            
            if should_stop:
                print(f"\nJudge intervening after round {round_num}")
                break
        
        # FINAL: Judge makes decision
        print(f"\n{'='*80}")
        print("JUDGE FINAL DECISION")
        print(f"{'='*80}\n")
        
        decision = self._make_final_judgment(
            product,
            round_1_args,
            conversation_history,
            clarification_answer
        )
        
        # Log judge decision
        scores_text = "\n".join([
           f"  • **{s.agent_name}**: {s.total_score:.2f}/5.0"
            for s in sorted(decision.scores, key=lambda x: x.total_score, reverse=True)
        ])
        
        num_rounds = len([a for a in all_arguments if a['type'] == 'critique_rebuttal']) // 3 + 1
        judge_msg = f"**FINAL VERDICT: {decision.final_label.upper()}**\n\n"
        judge_msg += f"After {num_rounds} rounds of deliberation...\n\n"
        judge_msg += f"**Scores:**\n{scores_text}\n\n"
        judge_msg += f"**Reasoning:**\n{decision.explanation}"
        
        self.add_to_log("Judge", judge_msg, self.judge.color)
        
        return decision, all_arguments
    
    def _make_initial_argument(self, advocate: AdvocateAgent, product: ProductInput) -> Argument:
        """Agent makes initial argument"""
        advocate.log(f"Making initial argument for {advocate.price_band}...")
        return advocate.build_argument(product)
    
    def _make_critique_and_rebuttal(
        self,
        advocate: AdvocateAgent,
        product: ProductInput,
        own_argument: Argument,
        other_arguments: List[Argument],
        conversation_history: List[Dict]
    ) -> str:
        """Agent critiques others and strengthens own position"""
        advocate.log(f"Critiquing opponents and reinforcing {advocate.price_band} position...")
        
        # Build context from conversation history
        history_text = "\n\n".join([
            f"[{entry['speaker']}]: {entry['message']}"
            for entry in conversation_history[-6:]  # Last 6 messages
        ])
        
        # Build critique prompt
        system_prompt = f"""You are {advocate.name}, defending your position that this product is {advocate.price_band.upper()}.

You've already made your case. Now you must:
1. CRITIQUE the arguments of other advocates
2. Point out flaws, weak evidence, or incorrect assumptions
3. REBUT their criticisms of your position
4. STRENGTHEN your own argument with new points

Be aggressive but factual. Attack weak reasoning, not agents personally.

Return valid json in this format:
{{
    "critique_of_opponents": "Explain flaws in other arguments",
    "rebuttal_to_criticisms": "Defend your position against implied criticisms",
    "strengthened_argument": "New evidence or reasoning for your position"
}}"""
        
        other_args_text = "\n\n".join([
            f"{arg.agent_name} argues for {arg.price_band.upper()}:\n{arg.reasoning}"
            for arg in other_arguments
        ])
        
        user_prompt = f"""CONVERSATION SO FAR:
{history_text}

YOUR POSITION: {advocate.price_band.upper()}
Your reasoning: {own_argument.reasoning}

OPPONENT ARGUMENTS:
{other_args_text}

Now critique their arguments and strengthen yours.
Respond with valid json only."""
        
        # Call LLM
        response = self.client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=TEMPERATURE,
            response_format={"type": "json_object"}
        )
        
        result = json.loads(response.choices[0].message.content)
        print(result)  # Debugging output
        # Format response
        critique_text = f"""**Critique of Opponents:**
{result['critique_of_opponents']}

**Rebuttal:**
{result['rebuttal_to_criticisms']}

**Strengthened Position:**
{result['strengthened_argument']}"""
        
        return critique_text
    
    def _check_judge_intervention(
        self,
        product: ProductInput,
        conversation_history: List[Dict],
        current_round: int
    ) -> bool:
        """Judge decides if debate should continue"""
        if current_round >= self.max_rounds:
            return True
        
        # Ask judge if they've seen enough
        history_text = "\n\n".join([
            f"[{entry['speaker']}]: {entry['message']}"
            for entry in conversation_history
        ])
        
        system_prompt = """You are the Judge monitoring a debate. 
        
Decide if you've seen enough arguments to make a decision, or if agents should continue debating.

Return valid json:
{{
    "should_stop": true/false,
    "reason": "Why you want to stop or continue"
}}

Stop if:
- Arguments are becoming repetitive
- Clear winner is emerging  
- No new substantive points being made

Continue if:
- Important points still being raised
- Significant disagreement remains
- New evidence appearing"""
        
        user_prompt = f"""DEBATE SO FAR:
{history_text}

Current round: {current_round}/{self.max_rounds}

Should I intervene now or let them continue?
Respond with valid json only."""
        
        response = self.client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.3,
            response_format={"type": "json_object"}
        )
        
        result = json.loads(response.choices[0].message.content)
        print(result)  # Debugging output
        
        if result["should_stop"]:
            self.add_to_log("Judge", f"Intervening: {result['reason']}", self.judge.color)
        
        return result["should_stop"]
    
    def _make_final_judgment(
        self,
        product: ProductInput,
        initial_arguments: List[Argument],
        conversation_history: List[Dict],
        clarification_answer: Optional[str]
    ) -> JudgeDecision:
        """Judge makes final decision considering full debate"""
        self.judge.log("Making final decision after reviewing full debate...")
        
        # Pass full conversation to judge
        return self.judge.evaluate_arguments(
            product,
            initial_arguments,
            clarification_answer
        )
    
    def add_to_log(self, agent: str, message: str, color: str):
        """Add message to debate log"""
        self.debate_log.append({
            "agent": agent,
            "message": message,
            "color": color
        })


print("Multi-round debate orchestrator defined")

## 10. Test Multi-Round Debate System

In [ ]:
# Initialize multi-round orchestrator
multi_orchestrator = MultiRoundDebateOrchestrator(openai_client, max_rounds=3)

# Use same test item
print(f"\n{'='*80}")
print("MULTI-ROUND DEBATE DEMONSTRATION")
print(f"{'='*80}\n")
print(f"Product: {test_product.title}")
print(f"Category: {test_product.category}")
print(f"Actual Price: ${test_item.price:.2f} ({get_price_band(test_item.price)})\n")

# Run multi-round debate
decision_multi, all_arguments = multi_orchestrator.run_multi_round_debate(test_product)

# Summary
print(f"\n{'='*80}")
print("MULTI-ROUND DEBATE SUMMARY")
print(f"{'='*80}\n")
print(f"Total Rounds: {max([a['round'] for a in all_arguments])}")
print(f"Total Exchanges: {len(all_arguments)}")
print(f"\nFinal Decision: {decision_multi.final_label.upper()}")
print(f"Actual Band: {get_price_band(test_item.price).upper()}")
print(f"Correct: {decision_multi.final_label == get_price_band(test_item.price)}")

# Show how arguments evolved
print(f"\n{'='*80}")
print("ARGUMENT EVOLUTION")
print(f"{'='*80}\n")

for round_num in range(1, max([a['round'] for a in all_arguments]) + 1):
    round_args = [a for a in all_arguments if a['round'] == round_num]
    print(f"\nROUND {round_num}: {len(round_args)} arguments")
    for arg in round_args:
        if arg['type'] == 'initial_argument':
            print(f"  - {arg['agent']}: Initial position ({arg['argument'].confidence:.0%} confidence)")
        else:
            print(f"  - {arg['agent']}: Critique & Rebuttal")

## 11. Interactive Gradio UI

In [ ]:
def format_debate_log(debate_log):
    """Format debate log for Gradio HTML display"""
    html_parts = []
    
    # Color mapping for HTML
    color_map = {
        Agent.GREEN: "#00ff00",
        Agent.YELLOW: "#ffff00",
        Agent.MAGENTA: "#ff00ff",
        Agent.CYAN: "#00ffff",
        "white": "#ffffff"
    }
    
    for entry in debate_log:
        color = color_map.get(entry["color"], "#ffffff")
        agent = entry["agent"]
        message = entry["message"].replace("\n", "<br>")
        
        html_parts.append(f"""
        <div style="margin: 15px 0; padding: 15px; background-color: #1a1a1a; border-left: 4px solid {color}; border-radius: 5px;">
            <div style="color: {color}; font-weight: bold; font-size: 16px; margin-bottom: 10px;">
                [{agent}]
            </div>
            <div style="color: #e0e0e0; line-height: 1.6;">
                {message}
            </div>
        </div>
        """)
    
    return "<div style='background-color: #0a0a0a; padding: 20px; border-radius: 10px;'>" + "".join(html_parts) + "</div>"


def run_classification_debate(
    title: str,
    category: str,
    brand: str,
    condition: str,
    description: str,
    clarification: str
):
    """Run debate for custom product input"""
    # Create product input
    product = ProductInput(
        title=title,
        category=category or "Unknown",
        brand=brand if brand else None,
        condition=condition if condition else None,
        description=description if description else None
    )
    
    # Run debate
    clarification_answer = clarification if clarification.strip() else None
    decision, arguments = orchestrator.run_debate(product, clarification_answer)
    
    # Format debate log
    debate_html = format_debate_log(orchestrator.debate_log)
    
    # Format decision summary
    summary = f"""## 🎯 Final Classification: **{decision.final_label.upper()}**

### 📊 Detailed Scores:
"""
    for score in sorted(decision.scores, key=lambda x: x.total_score, reverse=True):
        summary += f"""- **{score.agent_name}**: {score.total_score:.2f}/5.0
  - Evidence Grounding: {score.evidence_grounding:.2f}
  - Internal Consistency: {score.internal_consistency:.2f}
  - Domain Plausibility: {score.domain_plausibility:.2f}
  - Counterpoint Handling: {score.counterpoint_handling:.2f}
  - Confidence Calibration: {score.confidence_calibration:.2f}
"""
    
    summary += f"\n### 💡 Judge's Explanation:\n{decision.explanation}"
    
    if decision.clarification_question:
        summary += f"\n\n### Clarification Needed:\n{decision.clarification_question}"
    
    return debate_html, summary


def load_random_test_item():
    """Load random test item"""
    item = random.choice(test)
    return (
        item.title,
        item.category or "Unknown",
        item.brand or "",
        item.condition or "",
        item.description or "",
        f"Actual: ${item.price:.2f} ({get_price_band(item.price)})"
    )


# Create Gradio interface
with gr.Blocks(title="Multi-Agent Price Band Debate", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Multi-Agent Debate System for Price Band Classification
    
    Watch as three specialized advocate agents debate which price band a product belongs to,
    then see the Judge evaluate their arguments using a rigorous scoring rubric!
    
    **Price Bands:**
    - 💚 Budget: $0 - ${:.2f}
    - 💛 Midrange: ${:.2f} - ${:.2f}
    - 💜 Premium: ${:.2f}+
    """.format(BUDGET_MAX, BUDGET_MAX, MIDRANGE_MAX, MIDRANGE_MAX))
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## Product Input")
            
            title_input = gr.Textbox(
                label="Product Title",
                placeholder="e.g., Apple iPhone 13 Pro Max"
            )
            category_input = gr.Textbox(
                label="Category",
                placeholder="e.g., Cell Phones & Smartphones"
            )
            brand_input = gr.Textbox(
                label="Brand (optional)",
                placeholder="e.g., Apple"
            )
            condition_input = gr.Dropdown(
                label="Condition (optional)",
                choices=["", "New", "Used", "Refurbished", "For parts or not working"],
                value=""
            )
            description_input = gr.Textbox(
                label="Description (optional)",
                placeholder="Additional product details...",
                lines=3
            )
            clarification_input = gr.Textbox(
                label="Clarification Answer (if requested)",
                placeholder="Answer to judge's clarification question...",
                lines=2
            )
            
            with gr.Row():
                classify_btn = gr.Button("🎯 Run Debate", variant="primary")
                random_btn = gr.Button("🎲 Load Random Item")
            
            actual_info = gr.Textbox(label="Actual Info (for testing)", interactive=False)
        
        with gr.Column(scale=2):
            gr.Markdown("## 🗣️ Debate Transcript")
            debate_output = gr.HTML()
            
    gr.Markdown("## 📋 Decision Summary")
    summary_output = gr.Markdown()
    
    # Event handlers
    classify_btn.click(
        fn=run_classification_debate,
        inputs=[
            title_input,
            category_input,
            brand_input,
            condition_input,
            description_input,
            clarification_input
        ],
        outputs=[debate_output, summary_output]
    )
    
    random_btn.click(
        fn=load_random_test_item,
        inputs=[],
        outputs=[
            title_input,
            category_input,
            brand_input,
            condition_input,
            description_input,
            actual_info
        ]
    )
    

# Launch the app
demo.launch(inbrowser=True, share=False)